# WordPiece TOkenization for NER Task

In [ ]:
# ------------------------------------------------------------------------------------------------------------
#                      How to Add the Token Classification Head to DistillBERT
# ------------------------------------------------------------------------------------------------------------
# [https://huggingface.co/docs/transformers/model_doc/distilbert#transformers.DistilBertForTokenClassification]

from transformers import DistilBertTokenizer,DistilBertTokenizerFast, DistilBertModel, DistilBertForTokenClassification
import json

# NOTE:
#[https://github.com/huggingface/transformers/blob/v5.0.0rc0/src/transformers/models/distilbert/tokenization_distilbert.py#L23]
# it appears that DistilBertTokenizerFast, DistilBertTokenizer are now aliases

# ========================
# LOAD TOKENIZER AND MODEL
# ========================

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
# model = DistilBertModel.from_pretrained("distilbert-base-uncased")
# USE DistilBertForTokenClassification MODULE FOR NER TASKS
model = DistilBertForTokenClassification.from_pretrained("distilbert-base-uncased")
print("Hidden size:", model.config.dim)
# We add a classification layer to go from model.config.dim ==> number_labels

dataset = []
with open('data/synthetic_data_tokenized.jsonl', 'r') as f:
    for line in f:
        row = json.loads(line)
        dataset.append(json.loads(line))





/Users/robertagarcia/Desktop/learning/bert_symptom_ner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Hidden size: 768


# Tokenize Dataset using DistillBERT's Tokenizer

Convert your pre-tokenized whitespace tokens + BIO labels into model (wordpiece) tokens and produce aligned label ids per model token

## STEPS:
1. Give the white space tokens to the tokenizer
        - This will allow the proper assignment of the Labels to the subword tokens

2. Label each token (WordPiece tokenized) based on the words that these tokens belong to.

3. Convert the string labels to integer labels

### Quick Look at How it Works

In [7]:
# Load all data from JSONL into a list of dictionaries
ex1 = dataset[0]
print(ex1.keys())
print("TOkenize text")
toks = tokenizer(ex1['text'])
print(toks)

print("\nTokenize tokens")

print("with is_split_into_words=False")
toks_ids = tokenizer(ex1['word_tokens'],is_split_into_words=False) #?? DEFAULT = FALSE
print(toks_ids)
# Will not work because of nested lists
#print("Tokens:\n", tokenizer.convert_ids_to_tokens(toks_ids['input_ids']))

print("\nwith is_split_into_words=True")
toks_ids = tokenizer(ex1['word_tokens'],is_split_into_words=True) #??
print(toks_ids)
print("Word ids:\n", toks_ids.word_ids())
print("Tokens:\n", tokenizer.convert_ids_to_tokens(toks_ids['input_ids']))

# [CLS] -> Classification token (start of sequence)
# [SEP] -> Separator token (end of sequence)

# Tokenization flow: words -> word ids -> token ids

dict_keys(['text', 'word_tokens', 'labels', 'symptom_id', 'is_negated'])
TOkenize text
{'input_ids': [101, 2045, 2024, 2053, 8030, 1997, 2358, 14615, 2953, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Tokenize tokens
with is_split_into_words=False
{'input_ids': [[101, 2045, 102], [101, 2024, 102], [101, 2053, 102], [101, 8030, 102], [101, 1997, 102], [101, 2358, 14615, 2953, 102], [101, 1012, 102]], 'attention_mask': [[1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1]]}

with is_split_into_words=True
{'input_ids': [101, 2045, 2024, 2053, 8030, 1997, 2358, 14615, 2953, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Word ids:
 [None, 0, 1, 2, 3, 4, 5, 5, 5, 6, None]
Tokens:
 ['[CLS]', 'there', 'are', 'no', 'symptoms', 'of', 'st', '##rid', '##or', '.', '[SEP]']


# Tokenization

- assign -100 to special tokens: [CLS], [SEP], model does not need to learn the labels for this special tokens

Sources:
[]
[https://medium.com/@whyamit101/fine-tuning-bert-for-named-entity-recognition-ner-b42bcf55b51d]

In [2]:
ex = dataset[10]
# is_split_into_words=True: Input is pre-tokenized list of words, not a string
# - Tokenizer applies subword tokenization to each word individually
# - Returns word_ids() mapping each subword token back to its original word index
# - Critical for token classification: aligns word-level labels with subword tokens
# - Without it: tokenizer treats list as nested structure, causing errors
toks_ids = tokenizer(ex['word_tokens'],is_split_into_words=True)
print("Text:\n\t", ex['text'])
print("Input token ids:\n\t",toks_ids['input_ids'])
print("Word ids:\n\t", toks_ids.word_ids())
print("Tokens:\n\t", tokenizer.convert_ids_to_tokens(toks_ids['input_ids']))


Text:
	 Reports periumbilic pelvic lump.
Input token ids:
	 [101, 4311, 2566, 5007, 14454, 2594, 21877, 2140, 7903, 15116, 1012, 102]
Word ids:
	 [None, 0, 1, 1, 1, 1, 2, 2, 2, 3, 4, None]
Tokens:
	 ['[CLS]', 'reports', 'per', '##ium', '##bil', '##ic', 'pe', '##l', '##vic', 'lump', '.', '[SEP]']


In [8]:
# ===========================================================================
# Create wordpiece-tokenized dataset 
# ===========================================================================

import json

# Prepare list for storing tokenized samples (optional, for downstream use)
tokenized_samples = []

for row in dataset:

    # Tokenize word tokens for each sample
    words = row['word_tokens']
    toks_ids = tokenizer(words, is_split_into_words=True)
    input_ids = toks_ids['input_ids']
    word_ids = toks_ids.word_ids()

    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    token_labels = []
    previous_word_id = None

    for i, tok in enumerate(input_ids):
        word_id = word_ids[i]

        # Special tokens
        if word_id is None:
            token_labels.append("None") # -100
        # First subword of a word
        elif word_id != previous_word_id:
            token_labels.append(row['word_labels'][word_id])
        # Continuation subwords
        else:
            original_label = row['word_labels'][word_id]
            # If it's a B- tag, convert it to I-
            if isinstance(original_label, str) and original_label.startswith("B-"):
                fixed_label = original_label.replace("B-", "I-")
            else:
                fixed_label = original_label
            token_labels.append(fixed_label)
        previous_word_id = word_id

    # Prepare record for saving
    tokenized_sample = {
        "text": row["text"],
        "word_tokens": words,
        "word_labels": row["word_labels"],
        "tokens": tokens,
        "input_ids": input_ids,
        "token_labels": token_labels
    }
    tokenized_samples.append(tokenized_sample)

print("Example tokenized sample:")
print(tokenized_samples[0])


Example tokenized sample:
{'text': 'There are no symptoms of stridor.', 'word_tokens': ['There', 'are', 'no', 'symptoms', 'of', 'stridor', '.'], 'word_labels': ['O', 'O', 'O', 'O', 'O', 'B-SYMPTOM_s0486_NEG', 'O'], 'tokens': ['[CLS]', 'there', 'are', 'no', 'symptoms', 'of', 'st', '##rid', '##or', '.', '[SEP]'], 'input_ids': [101, 2045, 2024, 2053, 8030, 1997, 2358, 14615, 2953, 1012, 102], 'token_labels': ['None', 'O', 'O', 'O', 'O', 'O', 'B-SYMPTOM_s0486_NEG', 'I-SYMPTOM_s0486_NEG', 'I-SYMPTOM_s0486_NEG', 'O', 'None']}


In [9]:
with open("data_wordpiece_tokenized.jsonl", "w") as f:
    for sample in tokenized_samples:
        f.write(json.dumps(sample) + "\n")

## Convert String Labels to Integer Labels

In [10]:
import json
dataset = []
with open("data_wordpiece_tokenized.jsonl", "r") as f:
    for line in f:
        dataset.append(json.loads(line))

In [11]:
# Collect all unique labels from your dataset and saved it in jsons

unique_labels = set()

for row in dataset: 
    for lbl in row["token_labels"]:
        if lbl != "None":  # ignore special tokens
            unique_labels.add(lbl)

# Sort for stable ordering
unique_labels = sorted(list(unique_labels))

# Create mappings
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Number of labels:", len(label2id))

# Save mapping:
with open("label2id.json", "w") as f:
    json.dump(label2id, f, indent = 2)
with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent = 2)

Number of labels: 3499


In [12]:
# Convert labels to numeric IDs
for row in dataset:
    row["token_label_ids"] = [
        -100 if lbl == "None" else label2id[lbl]
        for lbl in row["token_labels"]
    ]

with open("data_wordpiece_tokenized.jsonl", "w") as f:
    for sample in dataset:
        f.write(json.dumps(sample) + "\n")

In [42]:
# Sanity check!

In [13]:
print(dataset[0]["token_labels"])
print(dataset[0]["token_label_ids"])

['None', 'O', 'O', 'O', 'O', 'O', 'B-SYMPTOM_s0486_NEG', 'I-SYMPTOM_s0486_NEG', 'I-SYMPTOM_s0486_NEG', 'O', 'None']
[-100, 3498, 3498, 3498, 3498, 3498, 970, 2722, 2722, 3498, -100]
